In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

from src.preprocessing import impute_missing, convert_types
from src.features import add_features, add_neighborhood_features, add_target_encoding_features, FEATURES
from src.utils import run_cv, make_submission

In [ ]:
train_df = pd.read_csv("../data/train.csv")
test_df  = pd.read_csv("../data/test.csv")

y = train_df["SalePrice"]
train_df = train_df.drop("SalePrice", axis=1)

df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

In [ ]:
# 前処理
df = impute_missing(df)
df = convert_types(df)

In [ ]:
# 特徴量生成
df = add_features(df)

train = df.iloc[:len(y)].copy()
test  = df.iloc[len(y):].copy()

train = add_neighborhood_features(train, train)
test  = add_neighborhood_features(test, train)

# KFold Target Encoding
train["SalePrice"] = y.values
train, test = add_target_encoding_features(train, test, target_col="SalePrice")
train = train.drop("SalePrice", axis=1)

X_train = train[FEATURES]
y_train = np.log1p(y)
X_test  = test[FEATURES]

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)

In [ ]:
params = {
    "boosting_type" : "gbdt",
    "objective"     : "regression",
    "metric"        : "rmse",
    "learning_rate" : 0.1,
    "num_leaves"    : 16,
    "n_estimators"  : 100000,
    "random_state"  : 123,
    "importance_type": "gain",
}

metrics, imp = run_cv(X_train, y_train, params)

In [ ]:
imp.head(20)

In [ ]:
# 全データで再学習 → submission
model = lgb.LGBMRegressor(**params)
model.fit(X_train, y_train)

make_submission(
    model, X_test, test["Id"],
    filepath="../submissions/hp-submission_03_feature_engineering.csv"
)